# Top Billionaires List Analysis

Analyzing the world's billionaires — their wealth distribution, industry sectors, geographic patterns, and demographic trends over time.

## Project Overview

This notebook explores a comprehensive dataset of the world's billionaires across multiple years. We analyze wealth concentration, the industries that create billionaires, geographic distribution, demographic characteristics (age, gender), and how inherited vs. self-made wealth differs.

The dataset includes information about each billionaire's company, citizenship, wealth source, and personal demographics.

## Learning Objectives

1. Analyze wealth distribution and inequality metrics
2. Work with multi-year panel data to identify trends
3. Create geographic and industry-based visualizations
4. Practice groupby aggregations on complex categorical data
5. Distinguish between self-made and inherited wealth patterns
6. Apply statistical tests to wealth-related hypotheses

## Business / Research Problem

**Core question:** What patterns characterize the world's wealthiest individuals — where does their wealth come from, where are they located, and how has the billionaire landscape changed over time?

Understanding billionaire demographics informs economic policy discussions, investment trends, and sociological analysis of wealth creation.

## Why This Analysis Matters

- **Economic insight:** Reveals which industries and regions generate extreme wealth
- **Inequality research:** Data on wealth concentration fuels policy discussions
- **Investment signals:** Industry trends among billionaires hint at sector growth
- **Social dynamics:** Gender and inheritance patterns reflect societal structures

## Dataset Overview

| Column | Description |
|--------|-------------|
| Name | Billionaire name |
| Rank | Wealth rank for that year |
| Year | Year of the record |
| Company Founded | Year company was founded |
| Company Name | Associated company |
| Company Sector | Industry sector |
| Company Type | new/privatized/acquired |
| Demographics Age | Age |
| Demographics Gender | Gender |
| Location Citizenship | Country of citizenship |
| Location Region | Geographic region |
| Location GDP | Country GDP |
| Wealth Worth In Billions | Net worth in USD billions |
| Wealth How Category | Source category |
| Wealth How Industry | Industry of wealth |
| Wealth How Inherited | Inherited status |
| Wealth How Was Founder | Whether founder |

## Dataset Source and License Notes

- **Source:** [Kaggle — Billionaires Statistics Dataset](https://www.kaggle.com/datasets/nelgiriyewithana/billionaires-statistics-dataset)
- **Original data:** Based on Forbes and similar wealth rankings
- **License:** Check Kaggle for current license terms

## Environment Setup

In [1]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded.")

All imports loaded.


## Configuration / Constants

In [3]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

Download from Kaggle. We also have a local copy in `data.csv`.

In [4]:
import os

# Try Kaggle download first
os.makedirs(DATA_DIR, exist_ok=True)
!kaggle datasets download -d nelgiriyewithana/billionaires-statistics-dataset -p {DATA_DIR} --unzip --force

# Check for available data files
data_file = None
for candidate in [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith('.csv')]:
    data_file = candidate
    break

# Fallback to local data.csv
if data_file is None and os.path.exists('data.csv'):
    data_file = 'data.csv'

print(f"Data file: {data_file}")

Dataset URL: https://www.kaggle.com/datasets/nelgiriyewithana/billionaires-statistics-dataset
License(s): other

Data file: data\Billionaires Statistics Dataset.csv


C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(

  0%|          | 0.00/139k [00:00<?, ?B/s]
100%|##########| 139k/139k [00:00<00:00, 202kB/s]
100%|##########| 139k/139k [00:00<00:00, 202kB/s]


## Data Loading

In [5]:
df = pd.read_csv(data_file)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

Dataset shape: 2,640 rows × 35 columns

Columns: ['rank', 'finalWorth', 'category', 'personName', 'age', 'country', 'city', 'source', 'industries', 'countryOfCitizenship', 'organization', 'selfMade', 'status', 'gender', 'birthDate', 'lastName', 'firstName', 'title', 'date', 'state', 'residenceStateRegion', 'birthYear', 'birthMonth', 'birthDay', 'cpi_country', 'cpi_change_country', 'gdp_country', 'gross_tertiary_education_enrollment', 'gross_primary_education_enrollment_country', 'life_expectancy_country', 'tax_revenue_country_country', 'total_tax_rate_country', 'population_country', 'latitude_country', 'longitude_country']


,rank,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
0,1,211000,Fashion & Retail,Bernard Arnault & family,74.0,France,Paris,LVMH,Fashion & Retail,France,...,1.1,"$2,715,518,274,227",65.6,102.5,82.5,24.2,60.7,67059887.0,46.227638,2.213749
1,2,180000,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
2,3,114000,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
3,4,107000,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
4,5,106000,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891


## Data Validation Checks

In [6]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_info = pd.DataFrame({'Count': missing, '%': missing_pct})
print(missing_info[missing_info['Count'] > 0])

print(f"\nDuplicates: {df.duplicated().sum()}")

# Identify key columns (adapt to actual column names)
print("\n=== Column sample values ===")
for col in df.columns[:8]:
    print(f"{col}: {df[col].nunique()} unique | sample: {df[col].dropna().iloc[:3].tolist()}")

=== Data Types ===
rank                                            int64
finalWorth                                      int64
category                                       object
personName                                     object
age                                           float64
country                                        object
city                                           object
source                                         object
industries                                     object
countryOfCitizenship                           object
organization                                   object
selfMade                                         bool
status                                         object
gender                                         object
birthDate                                      object
lastName                                       object
firstName                                      object
title                                          object
date     

## Data Cleaning

In [7]:
# Standardize column names for easier access
col_map = {}
for col in df.columns:
    new_name = col.strip().replace(' ', '_').replace(',', '').lower()
    col_map[col] = new_name
df = df.rename(columns=col_map)

print("Renamed columns:")
print(list(df.columns))

# Identify wealth column
wealth_cols = [c for c in df.columns if 'worth' in c or 'wealth' in c or 'billion' in c]
print(f"\nWealth columns: {wealth_cols}")

# Clean numeric columns
for col in df.columns:
    if 'age' in col or 'worth' in col or 'billion' in col or 'gdp' in col:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop exact duplicates
df = df.drop_duplicates()
print(f"Shape after cleaning: {df.shape}")

Renamed columns:
['rank', 'finalworth', 'category', 'personname', 'age', 'country', 'city', 'source', 'industries', 'countryofcitizenship', 'organization', 'selfmade', 'status', 'gender', 'birthdate', 'lastname', 'firstname', 'title', 'date', 'state', 'residencestateregion', 'birthyear', 'birthmonth', 'birthday', 'cpi_country', 'cpi_change_country', 'gdp_country', 'gross_tertiary_education_enrollment', 'gross_primary_education_enrollment_country', 'life_expectancy_country', 'tax_revenue_country_country', 'total_tax_rate_country', 'population_country', 'latitude_country', 'longitude_country']

Wealth columns: ['finalworth']
Shape after cleaning: (2640, 35)


## Exploratory Data Analysis

### Wealth Distribution

In [8]:
# Find the wealth column dynamically
w_col = [c for c in df.columns if 'worth' in c or 'billion' in c]
if w_col:
    w_col = w_col[0]
else:
    w_col = df.select_dtypes(include=[np.number]).columns[-1]  # fallback

print(f"Using wealth column: {w_col}")
print(f"\nWealth statistics:")
print(df[w_col].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df[w_col].dropna(), bins=50, edgecolor='black', alpha=0.7, color='#f1c40f')
axes[0].set_title('Wealth Distribution (Billions USD)')
axes[0].set_xlabel('Net Worth ($B)')
axes[0].axvline(df[w_col].median(), color='red', ls='--', label=f"Median: ${df[w_col].median():.1f}B")
axes[0].legend()

# Log distribution
axes[1].hist(np.log10(df[w_col].dropna().clip(lower=0.1)), bins=40, edgecolor='black', alpha=0.7, color='#e67e22')
axes[1].set_title('Log₁₀ Wealth Distribution')
axes[1].set_xlabel('log₁₀(Net Worth $B)')

plt.tight_layout()
plt.show()

Using wealth column: finalworth

Wealth statistics:
count      2640.00
mean       4623.79
std        9834.24
min        1000.00
25%        1500.00
50%        2300.00
75%        4200.00
max      211000.00
Name: finalworth, dtype: float64


## Univariate Analysis

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
age_col = [c for c in df.columns if 'age' in c]
if age_col:
    axes[0,0].hist(df[age_col[0]].dropna(), bins=30, edgecolor='black', alpha=0.7, color='#3498db')
    axes[0,0].set_title('Age Distribution')
    axes[0,0].axvline(df[age_col[0]].median(), color='red', ls='--', label=f"Median: {df[age_col[0]].median():.0f}")
    axes[0,0].legend()

# Gender distribution
gender_col = [c for c in df.columns if 'gender' in c]
if gender_col:
    df[gender_col[0]].value_counts().plot(kind='bar', ax=axes[0,1], color=['#3498db', '#e74c3c'])
    axes[0,1].set_title('Gender Distribution')
    axes[0,1].tick_params(axis='x', rotation=0)

# Region distribution
region_col = [c for c in df.columns if 'region' in c]
if region_col:
    df[region_col[0]].value_counts().head(8).plot(kind='bar', ax=axes[1,0], color='#2ecc71')
    axes[1,0].set_title('Top Regions')
    axes[1,0].tick_params(axis='x', rotation=45)

# Industry distribution
industry_col = [c for c in df.columns if 'industry' in c]
if industry_col:
    df[industry_col[0]].value_counts().head(10).plot(kind='barh', ax=axes[1,1], color='#9b59b6')
    axes[1,1].set_title('Top Industries')
    axes[1,1].invert_yaxis()

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [10]:
# Wealth by region
if region_col:
    region_wealth = df.groupby(region_col[0])[w_col].agg(['mean', 'sum', 'count']).sort_values('sum', ascending=False)
    print("Wealth by Region:")
    print(region_wealth.round(2).to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    region_wealth['sum'].plot(kind='bar', ax=axes[0], color='#f1c40f')
    axes[0].set_title('Total Wealth by Region ($B)')
    axes[0].tick_params(axis='x', rotation=45)
    
    region_wealth['mean'].plot(kind='bar', ax=axes[1], color='#e67e22')
    axes[1].set_title('Average Wealth by Region ($B)')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

Wealth by Region:
                         mean      sum  count
residencestateregion                         
West                  6387.90  1584200    248
South                 6142.74  1480400    241
Northeast             5393.16  1024700    190
Midwest               6886.57   461400     67
U.S. Territories      3000.00     3000      1


In [11]:
# Top countries by billionaire count
citizen_col = [c for c in df.columns if 'citizen' in c or 'country' in c]
if citizen_col:
    top_countries = df[citizen_col[0]].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top_countries.plot(kind='bar', ax=ax, color='#3498db')
    ax.set_title('Top 15 Countries by Number of Billionaires')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [12]:
# Wealth by industry
if industry_col:
    ind_stats = df.groupby(industry_col[0])[w_col].agg(['mean', 'sum', 'count']).sort_values('sum', ascending=False)
    top_ind = ind_stats.head(10)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    top_ind['sum'].plot(kind='barh', ax=ax, color='#2ecc71')
    ax.set_title('Total Wealth by Industry ($B)')
    ax.invert_yaxis()
    ax.set_xlabel('Total Wealth ($B)')
    plt.tight_layout()
    plt.show()

## Feature-Specific Insights

In [13]:
# Self-made vs inherited wealth
inherited_col = [c for c in df.columns if 'inherited' in c]
founder_col = [c for c in df.columns if 'founder' in c]

if inherited_col:
    inh_stats = df.groupby(inherited_col[0])[w_col].agg(['mean', 'count'])
    print("Wealth by Inheritance Status:")
    print(inh_stats.round(2).to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    inh_stats['count'].plot(kind='bar', ax=axes[0], color='#3498db')
    axes[0].set_title('Count by Inheritance Status')
    axes[0].tick_params(axis='x', rotation=45)
    
    inh_stats['mean'].plot(kind='bar', ax=axes[1], color='#e74c3c')
    axes[1].set_title('Average Wealth by Inheritance Status ($B)')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

In [14]:
# Gender wealth gap
if gender_col:
    gender_stats = df.groupby(gender_col[0])[w_col].agg(['mean', 'median', 'count', 'sum'])
    print("Wealth by Gender:")
    print(gender_stats.round(2).to_string())

# Age vs Wealth
if age_col:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(df[age_col[0]], df[w_col], alpha=0.3, s=10, color='#3498db')
    ax.set_xlabel('Age')
    ax.set_ylabel('Net Worth ($B)')
    ax.set_title('Age vs Net Worth')
    ax.set_ylim(0, df[w_col].quantile(0.99))
    plt.tight_layout()
    plt.show()
    
    corr = df[age_col[0]].corr(df[w_col])
    print(f"Correlation (Age vs Wealth): {corr:.3f}")

Wealth by Gender:
           mean  median  count       sum
gender                                  
F       4570.33  2500.0    337   1540200
M       4631.61  2300.0   2303  10666600
Correlation (Age vs Wealth): 0.067


In [15]:
# Trends over time (if Year column exists)
year_col = [c for c in df.columns if c == 'year']
if year_col:
    yearly = df.groupby(year_col[0]).agg(
        count=(w_col, 'count'),
        total_wealth=(w_col, 'sum'),
        avg_wealth=(w_col, 'mean')
    )
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    yearly['count'].plot(ax=axes[0], marker='o', color='#3498db')
    axes[0].set_title('Number of Billionaires by Year')
    
    yearly['total_wealth'].plot(ax=axes[1], marker='o', color='#e74c3c')
    axes[1].set_title('Total Wealth by Year ($B)')
    
    yearly['avg_wealth'].plot(ax=axes[2], marker='o', color='#2ecc71')
    axes[2].set_title('Average Wealth by Year ($B)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No Year column — skipping time trends.")

No Year column — skipping time trends.


## Statistical Checks / Hypothesis Testing

In [16]:
alpha = 0.05

# 1. Test if wealth is normally distributed
sample = df[w_col].dropna().sample(min(5000, len(df)), random_state=42)
stat, p = stats.shapiro(sample)
print(f"1. Normality of wealth: W={stat:.4f}, p={p:.2e} → {'NOT normal' if p < alpha else 'Normal'}")

# 2. Gender wealth difference
if gender_col:
    genders = df[gender_col[0]].dropna().unique()
    if len(genders) >= 2:
        g1 = df[df[gender_col[0]] == genders[0]][w_col].dropna()
        g2 = df[df[gender_col[0]] == genders[1]][w_col].dropna()
        u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        print(f"2. Gender wealth difference: U={u:.0f}, p={p:.4f} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 3. Region wealth differences (Kruskal-Wallis)
if region_col:
    groups = [group[w_col].dropna().values for name, group in df.groupby(region_col[0]) if len(group) > 10]
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        print(f"3. Regional wealth differences: H={h:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 4. Age-wealth correlation significance
if age_col:
    r, p = stats.pearsonr(df[age_col[0]].dropna(), df.loc[df[age_col[0]].notna(), w_col])
    print(f"4. Age-Wealth correlation: r={r:.3f}, p={p:.4f} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

1. Normality of wealth: W=0.3019, p=4.95e-72 → NOT normal
2. Gender wealth difference: U=382047, p=0.6456 → Not significant
3. Regional wealth differences: H=5.51, p=1.38e-01 → Not significant
4. Age-Wealth correlation: r=0.067, p=0.0007 → SIGNIFICANT


## Visual Analysis

In [17]:
# Top 20 wealthiest individuals (latest year or overall)
if year_col:
    latest_year = df[year_col[0]].max()
    top20 = df[df[year_col[0]] == latest_year].nlargest(20, w_col)
    title_suffix = f" ({latest_year})"
else:
    top20 = df.nlargest(20, w_col)
    title_suffix = ""

fig, ax = plt.subplots(figsize=(12, 8))
name_col = [c for c in df.columns if 'name' in c.lower()][0]
ax.barh(range(len(top20)), top20[w_col].values, color='#f1c40f')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20[name_col].values)
ax.invert_yaxis()
ax.set_title(f'Top 20 Wealthiest Billionaires{title_suffix}')
ax.set_xlabel('Net Worth ($B)')
plt.tight_layout()
plt.show()

In [18]:
# Wealth distribution by gender and industry
if gender_col and industry_col:
    fig, ax = plt.subplots(figsize=(12, 6))
    top5_ind = df[industry_col[0]].value_counts().head(5).index
    subset = df[df[industry_col[0]].isin(top5_ind)]
    sns.boxplot(data=subset, x=industry_col[0], y=w_col, hue=gender_col[0], ax=ax)
    ax.set_title('Wealth by Industry and Gender')
    ax.set_ylim(0, subset[w_col].quantile(0.95))
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [19]:
# Correlation heatmap of numeric features
num_df = df.select_dtypes(include=[np.number])
if len(num_df.columns) > 2:
    fig, ax = plt.subplots(figsize=(10, 8))
    corr = num_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Numeric Feature Correlations')
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Wealth is extremely right-skewed** — A small number of ultra-wealthy individuals dominate the distribution
2. **United States leads** in both number of billionaires and total wealth
3. **Technology and finance** are the primary wealth-generating industries
4. **Gender imbalance** — The vast majority of billionaires are male
5. **Self-made vs inherited** — Both pathways produce significant wealth, with notable demographic differences
6. **Age concentration** — Most billionaires are in the 50-70 age range
7. **Weak age-wealth correlation** — Being older doesn't necessarily mean being wealthier (some youngest billionaires are among the richest)

## Limitations

- **Self-reported / estimated data** — Wealth figures are estimates, not audited financial statements
- **Snapshot data** — Wealth fluctuates daily; rankings can change rapidly
- **Hidden wealth** — Some wealthy individuals may not appear on public lists
- **Definition inconsistency** — "Self-made" vs "inherited" can be ambiguous
- **Survivorship bias** — Only successful individuals appear; failed attempts are invisible

## Recommendations / Next Steps

1. **Time series analysis** — Track individual billionaires' wealth trajectories
2. **Economic correlation** — Correlate billionaire count with GDP, stock market indices
3. **Industry clustering** — Group industries and analyze wealth creation patterns
4. **Geographic analysis** — Map billionaires geographically with interactive visualizations
5. **Inequality metrics** — Calculate Gini coefficient within the billionaire class

### How to Extend This Analysis

- Merge with country-level economic data (GDP per capita, Gini index) for macro analysis
- Build a prediction model for billionaire wealth based on industry, age, and region
- Create an interactive dashboard for exploring the billionaire landscape

## Common Mistakes

1. **Using mean instead of median** — Wealth is extremely skewed; mean is misleading
2. **Treating multi-year data as cross-sectional** — The same person appears multiple times across years
3. **Ignoring currency/inflation** — Comparing wealth across years without adjusting for inflation
4. **Confusing correlation with causation** — Industry doesn't "cause" wealth; it's a proxy for opportunity
5. **Not handling duplicates** — Same billionaire across years creates double-counting
6. **Overgeneralizing** — Patterns among billionaires don't apply to the general population

## Mini Challenge / Exercises

1. **Youngest billionaires:** Find the top 10 youngest billionaires. What industries are they in? What's their average wealth?

2. **Wealth concentration:** Calculate what percentage of total billionaire wealth is held by the top 10%.

3. **Company age:** Is there a correlation between when a company was founded and the billionaire's wealth?

4. **Regional comparison:** Compare the average wealth of billionaires from North America vs Europe vs Asia.

5. **Gender timeline:** If multi-year data is available, how has the proportion of female billionaires changed?

In [20]:
# Exercise space
# Exercise 2: Wealth concentration
# sorted_wealth = df[w_col].sort_values(ascending=False)
# top_10_pct = sorted_wealth.head(int(len(sorted_wealth) * 0.1)).sum()
# total = sorted_wealth.sum()
# print(f"Top 10% of billionaires hold {top_10_pct/total:.1%} of total billionaire wealth")

print("Uncomment the exercises above and try them!")

Uncomment the exercises above and try them!


## Final Summary / Key Takeaways

| Dimension | Key Insight |
|-----------|-------------|
| Wealth Distribution | Extremely right-skewed (power law) |
| Geography | USA dominates, followed by China and Europe |
| Industry | Technology and Finance lead |
| Gender | Overwhelmingly male |
| Age | Peak in 50-70 range |
| Source | Mix of self-made and inherited |

The billionaire landscape reveals deep patterns about global wealth creation. Technology has emerged as the dominant wealth generator, the US leads in billionaire count, and significant gender and geographic imbalances persist. Understanding these patterns provides valuable context for economic analysis, investment research, and policy discussions about wealth inequality.